# Lab 2 · 정기구학과 작업 영역

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/djfksjd/robot_study/blob/main/notebooks/01_forward_kinematics_workspace.ipynb)

목표
1. 관절값 `q = (θ1, θ2, d3, θ4)` 로 TCP 위치 `(x, y, z, yaw)` 를 계산한다.
2. 팔을 그림으로 그려 식과 그림이 맞는지 확인한다.
3. 관절 한계 안에서 닿을 수 있는 영역(작업 영역)을 점으로 그린다.

막히면 `prompts/03_forward_kinematics.md` 의 프롬프트를 AI 챗에 붙여넣는다.

In [ ]:
# 0) 저장소 준비 — Colab에서는 처음 한 번 clone, 로컬에서는 저장소 루트에서 실행하면 건너뛴다
import os, sys
if not os.path.exists("model/scara_params.json"):
    if not os.path.exists("robot_study"):
        !git clone -q https://github.com/djfksjd/robot_study.git
    %cd robot_study
sys.path.insert(0, "python")
print("작업 폴더:", os.getcwd())

In [ ]:
import json, math
import numpy as np
import matplotlib.pyplot as plt
from scara_kinematics import Scara

robot = Scara.load()
print(json.dumps(json.load(open("model/scara_params.json", encoding="utf-8"))["limits"], indent=1))
print("L1 =", robot.L1, "mm,  L2 =", robot.L2, "mm")

## 1. 정기구학 식

$x = L_1\cos\theta_1 + L_2\cos(\theta_1+\theta_2)$,
$y = L_1\sin\theta_1 + L_2\sin(\theta_1+\theta_2)$,
$z = z_0 - d_3$, $\;\text{yaw} = \theta_1+\theta_2+\theta_4$

In [ ]:
def show(q_deg_mm):
    t1, t2, d3, t4 = q_deg_mm
    q = (math.radians(t1), math.radians(t2), d3, math.radians(t4))
    x, y, z, yaw = robot.fk(q)
    ex, ey = robot.elbow_xy(q)
    print(f"q = {q_deg_mm}  ->  TCP x={x:.1f} y={y:.1f} z={z:.1f} mm, yaw={math.degrees(yaw):.1f}°")
    return (0, ex, x), (0, ey, y)

fig, ax = plt.subplots(figsize=(5, 5))
for q, c in [((0, 0, 0, 0), "k"), ((30, 45, 50, 0), "tab:blue"), ((90, -90, 100, 0), "tab:orange"), ((-60, 120, 0, 0), "tab:green")]:
    xs, ys = show(q)
    ax.plot(xs, ys, "-o", color=c, lw=3, label=str(q))
ax.set_aspect("equal"); ax.grid(True); ax.legend(fontsize=8); ax.set_xlabel("x (mm)"); ax.set_ylabel("y (mm)")
ax.set_title("SCARA top view (Z-up)")  # 그래프 글자는 Colab 한글 폰트 문제로 영어; plt.show()

## 2. 실습 과제
- `(θ1, θ2) = (0, 90)` 일 때 TCP 좌표를 **손으로** 먼저 계산하고 코드 결과와 비교하라.
- `θ4` 를 바꾸면 x, y 는 변하는가? yaw 는?

In [ ]:
# 직접 바꿔 보기
show((0, 90, 0, 0)); show((0, 90, 0, 45));

## 3. 작업 영역 — 관절 공간을 무작위로 뽑아 FK로 찍는다

In [ ]:
rng = np.random.default_rng(0)
lim = robot.limits
t1 = np.radians(rng.uniform(*lim["theta1_deg"], 20000))
t2 = np.radians(rng.uniform(*lim["theta2_deg"], 20000))
x = robot.L1 * np.cos(t1) + robot.L2 * np.cos(t1 + t2)
y = robot.L1 * np.sin(t1) + robot.L2 * np.sin(t1 + t2)

fig, ax = plt.subplots(figsize=(6, 6))
ax.scatter(x, y, s=1, alpha=0.3)
for r in (robot.reach_min, robot.reach_max):
    ax.add_patch(plt.Circle((0, 0), r, fill=False, ls="--", color="r"))
scene = json.load(open("model/task_scene.json", encoding="utf-8"))
for n, o in scene["objects"].items():
    ax.plot(o["x"], o["y"], "s", ms=8, color=o["color"]); ax.annotate(n, (o["x"], o["y"]))
for n, t in scene["targets"].items():
    ax.plot(t["x"], t["y"], "*", ms=14, color="gold"); ax.annotate(n, (t["x"], t["y"]))
ax.set_aspect("equal"); ax.grid(True)
ax.set_title(f"Workspace: r {robot.reach_min:.0f}~{robot.reach_max:.0f} mm, theta1 {lim['theta1_deg']} deg")
plt.show()

## 확인 질문 (AI 챗에 물어보고, 답을 코드로 검증)
1. 작업 영역 안쪽 구멍(반경 약 143 mm)은 왜 생기는가?
2. 로봇 뒤쪽(x<0, |y| 작음)이 비는 이유는?
3. 같은 (x, y)에 팔꿈치 해가 2개인 이유는? → Lab 3